# Урок 13. Визуализация данных

**Дата:** 17.12.2026 · **60 минут** · 8–9 класс  
**Модуль 3.** Данные (уроки 10–16)  
**Опирается на:** урок 10 (pandas), урок 12 (разведка: `groupby`, корреляция).

Таблица умеет «молчать»: `describe()` выглядит нормально, а на графике видно две группы, дыру или один безумный выброс. **График — это проверка данных глазами** и способ объяснить вывод другому человеку. Сегодня строим четыре главных вида графиков и учимся под каждым писать **вывод**.

## Цели урока

К концу занятия ты сможешь:

1. Выбирать вид графика под вопрос.
2. Строить гистограмму, столбчатую диаграмму, диаграмму рассеяния и линейный график.
3. Подписывать заголовок и оси.
4. Формулировать вывод из графика одной фразой.
5. Замечать графики, которые обманывают масштабом.

## План на 60 минут

| Минуты | Что делаем |
|-------:|------------|
| 0–5 | Какой график для какого вопроса |
| 5–35 | Гистограмма, столбцы, рассеяние, линия |
| 35–40 | Честные графики и графики-детекторы ошибок |
| 40–55 | Практика: 4 графика с выводами |
| 55–60 | Итог и домашнее задание |

## 1. Какой график для какого вопроса

| Вопрос | График | Пример |
|--------|--------|--------|
| Как распределены значения одного числового столбца? | **гистограмма** (`hist`) | какие баллы встречаются чаще |
| Сколько в каждой категории? Чему равно среднее по группам? | **столбчатая** (`bar`) | учеников в каждом кружке |
| Связаны ли два числовых столбца? | **рассеяние** (`scatter`) | часы и балл |
| Как меняется значение по порядку (дни, шаги)? | **линия** (`plot`) | средний балл в зависимости от часов |

**Правило:** один график — один вопрос — один вывод.

## 2. Данные и библиотека

Рисовать будем библиотекой **matplotlib** — основной библиотекой графиков в Python. Её модуль `pyplot` принято подключать под коротким именем `plt`.

Данные — те же 60 учеников, что на уроке 12 (та же функция-генератор с тем же зерном).

In [ ]:
import random
import pandas as pd
import matplotlib.pyplot as plt

def make_students(n, seed):
    """Сгенерировать учебную таблицу учеников (как на уроке 12)."""
    random.seed(seed)
    rows = []
    for i in range(n):
        hours = random.randint(1, 14)
        attendance = min(100, max(20, round(50 + 3 * hours + random.gauss(0, 10))))
        score = min(100, max(0, round(20 + 4 * hours + 0.3 * attendance + random.gauss(0, 8))))
        rows.append({
            "класс": random.choice([8, 9]),
            "кружок": random.choice(["Python", "Роботы", "Игры"]),
            "часы": hours,
            "посещаемость": attendance,
            "сон": round(random.uniform(6, 9.5), 1),
            "балл": score,
        })
    return pd.DataFrame(rows)

df = make_students(60, seed=12)
df["сдал"] = df["балл"] >= 60
df.head()

Каждый график строим по одной схеме:

```python
plt.figure(figsize=(7, 4))     # новый «холст»: ширина и высота
plt.hist(...)                  # сам график
plt.title("Заголовок")         # о чём график
plt.xlabel("Ось X")            # что по горизонтали
plt.ylabel("Ось Y")            # что по вертикали
plt.show()                     # показать
```

## 3. Гистограмма: как распределены значения

Гистограмма делит числа на **корзины** (интервалы) и показывает, сколько значений попало в каждую. Параметр `bins` — число корзин.

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df["балл"], bins=10, edgecolor="black")
plt.title("Распределение итоговых баллов")
plt.xlabel("Балл")
plt.ylabel("Число учеников")
plt.show()

**Вывод (пример):** «Баллы встречаются по всей шкале от низких до 100; заметная группа учеников набрала максимум — возможно, шкала для них слишком лёгкая».

**Что искать на гистограмме:**
- один «холм» или несколько (смешаны разные группы?);
- длинный «хвост» в одну сторону;
- подозрительные столбики на краях — у 0 или 100;
- одинокие значения далеко от остальных — выбросы.

**Попробуй сам:** поменяй `bins` на 4 и на 30. При слишком малом числе корзин детали пропадают, при слишком большом — картинка становится «нервной».

## 4. Столбчатая диаграмма: категории и группы

Для категорий сначала считаем частоты (`value_counts`) или средние по группам (`groupby`), потом рисуем столбцы.

In [ ]:
counts = df["кружок"].value_counts()

plt.figure(figsize=(6, 4))
plt.bar(counts.index, counts.values)
plt.title("Сколько учеников в каждом кружке")
plt.xlabel("Кружок")
plt.ylabel("Число учеников")
plt.show()

`counts.index` — названия категорий (подписи столбцов), `counts.values` — сами числа (высота столбцов).

Средние по группам из урока 12 тоже удобно показывать столбцами:

In [ ]:
share = df.groupby("кружок")["сдал"].mean()

plt.figure(figsize=(6, 4))
plt.bar(share.index, share.values, color="seagreen")
plt.title("Доля сдавших по кружкам")
plt.xlabel("Кружок")
plt.ylabel("Доля сдавших")
plt.ylim(0, 1)
plt.show()

`plt.ylim(0, 1)` задаёт пределы оси Y. Для долей честно показывать всю шкалу от 0 до 1 — почему, увидим в разделе 7.

**Гистограмма или столбчатая?** Гистограмма сама режет **число** на корзины. Столбчатая показывает **готовые категории**. Гистограмма по названиям кружков смысла не имеет.

## 5. Диаграмма рассеяния: связь двух чисел

Каждый ученик — точка: по горизонтали часы, по вертикали балл. Если облако точек вытянуто вверх-вправо — связь положительная (корреляция из урока 12).

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(df["часы"], df["балл"], alpha=0.6)
plt.title("Часы занятий и итоговый балл")
plt.xlabel("Часов в неделю")
plt.ylabel("Балл")
plt.grid(alpha=0.3)
plt.show()

- `alpha=0.6` — прозрачность точек: там, где точки наложились, цвет гуще.
- `figsize=(6, 5)` — размер рисунка (ширина, высота); `color` — цвет, `edgecolor` — цвет рамки столбиков; `marker="o"` — кружки на точках линии.
- `plt.grid` — сетка, чтобы легче считывать значения.

**Вывод:** «Чем больше часов, тем выше балл: облако точек вытянуто вверх-вправо. При 10 часах и больше почти все набирают 80+».

Сравни со сном — тут корреляция была почти нулевой:

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(df["сон"], df["балл"], alpha=0.6, color="orange")
plt.title("Сон и итоговый балл")
plt.xlabel("Часов сна")
plt.ylabel("Балл")
plt.grid(alpha=0.3)
plt.show()

Облако без направления — связи не видно.

### Цвет как третий столбец

Можно раскрасить точки по категории: например, сдавшие — одним цветом, несдавшие — другим. Для этого рисуем две группы по очереди и добавляем **легенду**.

In [ ]:
passed = df[df["сдал"]]
failed = df[~df["сдал"]]

plt.figure(figsize=(6, 5))
plt.scatter(passed["часы"], passed["посещаемость"], label="сдал", color="seagreen", alpha=0.7)
plt.scatter(failed["часы"], failed["посещаемость"], label="не сдал", color="crimson", alpha=0.7)
plt.title("Часы, посещаемость и результат")
plt.xlabel("Часов в неделю")
plt.ylabel("Посещаемость, %")
plt.legend()
plt.show()

На этом графике видно, что сдавшие и несдавшие почти разделяются по часам. Запомни эту картинку: на уроках про машинное обучение модель будет искать как раз такую **границу между группами**.

## 6. Линейный график: значение по порядку

Линия подходит, когда по горизонтали **упорядоченная** величина: дни, номера попыток, часы. Посчитаем средний балл для каждого количества часов и соединим точки линией.

In [ ]:
by_hours = df.groupby("часы")["балл"].mean()

plt.figure(figsize=(7, 4))
plt.plot(by_hours.index, by_hours.values, marker="o")
plt.title("Средний балл в зависимости от часов занятий")
plt.xlabel("Часов в неделю")
plt.ylabel("Средний балл")
plt.grid(alpha=0.3)
plt.show()

## 7. Честные и нечестные графики

Одни и те же числа можно показать так, что разница покажется огромной. Сравни два графика — данные одинаковые, разные только пределы оси Y.

`plt.subplots(1, 2)` создаёт **два графика рядом** (1 ряд, 2 графика). Функция возвращает сразу **два** значения, поэтому слева от `=` две переменные: `fig` — весь рисунок, `axes` — список графиков: `axes[0]` — левый, `axes[1]` — правый. У каждого графика свои команды с приставкой `set_`: `set_title`, `set_ylim`.

In [ ]:
means = df.groupby("класс")["балл"].mean()
labels = ["8 класс", "9 класс"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(labels, means.values)
axes[0].set_ylim(0, 100)
axes[0].set_title("Честно: шкала от 0 до 100")

axes[1].bar(labels, means.values)
axes[1].set_ylim(means.min() - 2, means.max() + 2)
axes[1].set_title("Обманчиво: шкала обрезана")
plt.show()

print(means.round(1))

На правом графике кажется, что один класс учится в разы лучше. На самом деле разница — несколько баллов. **Обрезанная шкала** — частый приём рекламы и плохих отчётов. Инженер показывает честно и подписывает числа.

## 8. График как детектор ошибок

Графики помогают найти грязь, которую мы чистили на уроке 11. Добавим в копию данных одну ошибку — балл 300 — и посмотрим.

`broken.loc[0, "балл"] = 300` — запись в одну ячейку: строка с индексом 0, столбец «балл».

In [ ]:
broken = df.copy()
broken.loc[0, "балл"] = 300      # специально портим одно значение

plt.figure(figsize=(6, 4))
plt.scatter(broken["часы"], broken["балл"], alpha=0.6)
plt.title("Одна ошибка в данных сразу видна")
plt.xlabel("Часов в неделю")
plt.ylabel("Балл")
plt.show()

Точка далеко над всеми остальными — это выброс. В `describe()` его тоже видно (максимум 300), но на графике — мгновенно.

## 9. Сохранить график в файл

Для отчёта или презентации проекта график сохраняют в картинку командой `plt.savefig` (до `plt.show()`). Файл появится в панели файлов Colab — оттуда его можно скачать.

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(df["часы"], bins=7, edgecolor="black")
plt.title("Сколько часов в неделю занимаются ученики")
plt.xlabel("Часов в неделю")
plt.ylabel("Число учеников")
plt.savefig("hours_hist.png", dpi=120)
plt.show()

## 10. Чек-лист «график готов»

- [ ] вид графика подходит к вопросу;
- [ ] есть заголовок, который отвечает на вопрос «о чём это»;
- [ ] подписаны обе оси (что и в каких единицах);
- [ ] шкала не обманывает;
- [ ] под графиком — **вывод одним предложением**, а не «получился график».

| Плохой вывод | Хороший вывод |
|--------------|---------------|
| «Красивый график» | «Чем больше часов, тем выше балл» |
| «Тут точки» | «Связи сна и балла не видно: облако без направления» |
| «Роботы лучше» | «Доля сдавших в Роботах выше (24 ученика), но там и занимаются больше часов» |

## 11. Практика: 4 графика с выводами

Для каждого графика — заголовок, подписи осей и вывод в `print` одной фразой.

In [ ]:
# График 1. Гистограмма посещаемости. Где «основная масса»? Есть ли хвост?

In [ ]:
# График 2. Столбчатая: средний балл по кружкам (groupby + bar), шкала от 0 до 100.

In [ ]:
# График 3. Рассеяние: посещаемость и балл. Есть ли связь?

In [ ]:
# График 4. Столбчатая: сколько сдавших и несдавших в каждом классе.
# Подсказка: pd.crosstab(df["класс"], df["сдал"]) и метод .plot(kind="bar") у таблицы

<details>
<summary>Пример решений</summary>

```python
plt.figure(figsize=(6, 4))
plt.hist(df["посещаемость"], bins=8, edgecolor="black")
plt.title("Распределение посещаемости")
plt.xlabel("Посещаемость, %")
plt.ylabel("Число учеников")
plt.show()
print("Вывод 1: большинство ходит на 60–90% занятий, есть небольшая группа с низкой посещаемостью.")

avg = df.groupby("кружок")["балл"].mean()
plt.figure(figsize=(6, 4))
plt.bar(avg.index, avg.values)
plt.ylim(0, 100)
plt.title("Средний балл по кружкам")
plt.xlabel("Кружок")
plt.ylabel("Средний балл")
plt.show()
print("Вывод 2: различия между кружками есть, но вспомни загадку урока 12 — дело в часах.")

plt.figure(figsize=(6, 5))
plt.scatter(df["посещаемость"], df["балл"], alpha=0.6)
plt.title("Посещаемость и балл")
plt.xlabel("Посещаемость, %")
plt.ylabel("Балл")
plt.show()
print("Вывод 3: чем выше посещаемость, тем выше балл, но разброс больше, чем у часов.")

table = pd.crosstab(df["класс"], df["сдал"])
table.plot(kind="bar", rot=0)
plt.title("Сдавшие и несдавшие по классам")
plt.xlabel("Класс")
plt.ylabel("Число учеников")
plt.show()
print("Вывод 4: в обоих классах сдавших больше, чем несдавших.")
```
</details>

## Работа с ИИ-помощником

- «Как добавить подписи значений над столбцами в `plt.bar`?»
- «Какой график лучше подходит, чтобы сравнить распределение баллов в двух классах?»
- Плохо: «Нарисуй все графики и напиши выводы». Вывод — это твоё понимание данных.

## Мини-глоссарий урока 13

| Термин | Смысл |
|--------|-------|
| **matplotlib / `plt`** | библиотека графиков |
| **Гистограмма** | сколько значений попало в каждую корзину |
| **Столбчатая диаграмма** | высота столбца = число для категории |
| **Диаграмма рассеяния** | точки по двум числовым осям |
| **Легенда** | подписи, какой цвет что значит |
| **Обрезанная шкала** | ось не от нуля — разница кажется больше |

## Итог урока

1. Вид графика выбирают по вопросу: распределение — гистограмма, категории — столбцы, связь — рассеяние, порядок — линия.
2. У графика есть заголовок, подписи осей и честная шкала.
3. Под каждым графиком — вывод одной фразой.

## Домашнее задание (20 минут)

Построй **два** графика разных видов по таблице «Журнал кружков» из урока 10 (её можно вставить из того ноутбука). Под каждым — вывод одним предложением.

In [ ]:
# Домашнее задание

## Самопроверка

1. Какой график подойдёт для вопроса «сколько учеников в каждом классе»?
2. Чем гистограмма отличается от столбчатой диаграммы?
3. Что показывает облако точек, вытянутое вверх-вправо?
4. Почему обрезанная шкала может обмануть?
5. Чем плох вывод «получился красивый график»?

**Следующий урок (24.12.2026):** практика по всему модулю данных — от сырой таблицы до отчёта с графиками.